# Multimodal Image Evaluation with LangSmith

This notebook demonstrates two approaches for sending images to Claude via a LangSmith dataset and experiment flow, evaluated with an LLM-as-judge.

| Approach | Function | Image size limit | Extra API calls | Beta required |
|---|---|---|---|---|
| **Presigned URL** | `target_presigned_url` | ~5 MB | None | No |
| **Files API** | `target_files_api` | 500 MB | Upload + delete per image | Yes |

## Setup

In [1]:
import uuid
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

import anthropic
from langsmith import Client

In [2]:
IMAGES_DIR = Path("images")
DATASET_NAME = "multimodal-image-descriptions"

EXAMPLES = [
    {
        "input": "Describe this image in detail.",
        "image_file": "city_skyline.jpg",
        "reference_output": "A city skyline with tall buildings and urban architecture.",
    },
    {
        "input": "What is the main subject of this image?",
        "image_file": "mountain_landscape.jpg",
        "reference_output": "A mountain landscape with snow-capped peaks and dramatic scenery.",
    },
    {
        "input": "Describe the mood and atmosphere of this image.",
        "image_file": "ocean_waves.jpg",
        "reference_output": "An ocean scene with waves, conveying a calm or powerful natural atmosphere.",
    },
    {
        "input": "What colors dominate this image?",
        "image_file": "forest_trail.jpg",
        "reference_output": "A forest trail dominated by green foliage and earthy brown tones.",
    },
    {
        "input": "Describe the lighting and time of day shown in this image.",
        "image_file": "desert_sunset.jpg",
        "reference_output": "A desert landscape during sunset with warm golden and orange lighting.",
    },
]

## Create Dataset

Upload images as LangSmith attachments (up to 20 MB each).

In [3]:
ls_client = Client()

# Delete existing dataset if it exists
try:
    existing = ls_client.read_dataset(dataset_name=DATASET_NAME)
    ls_client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset: {DATASET_NAME}")
except Exception:
    pass

dataset = ls_client.create_dataset(
    dataset_name=DATASET_NAME,
    description="Dataset for evaluating multimodal image description quality",
)
print(f"Created dataset: {DATASET_NAME} (id={dataset.id})")

examples = []
for ex in EXAMPLES:
    image_path = IMAGES_DIR / ex["image_file"]
    image_bytes = image_path.read_bytes()
    print(f"  Adding example: {ex['image_file']} ({len(image_bytes) / 1024 / 1024:.1f} MB)")

    examples.append({
        "id": uuid.uuid4(),
        "inputs": {
            "input": ex["input"],
            "image_file": ex["image_file"],
        },
        "outputs": {
            "answer": ex["reference_output"],
        },
        "attachments": {
            "image": {
                "mime_type": "image/jpeg",
                "data": image_bytes,
            },
        },
    })

ls_client.create_examples(dataset_id=dataset.id, examples=examples)
print(f"Added {len(examples)} examples with image attachments")

Deleted existing dataset: multimodal-image-descriptions
Created dataset: multimodal-image-descriptions (id=a2491b2a-d44d-4b87-bd61-8c9051d24383)
  Adding example: city_skyline.jpg (5.2 MB)
  Adding example: mountain_landscape.jpg (4.7 MB)
  Adding example: ocean_waves.jpg (3.5 MB)
  Adding example: forest_trail.jpg (4.2 MB)
  Adding example: desert_sunset.jpg (5.2 MB)
Added 5 examples with image attachments


## Target Functions

### Approach 1: Presigned URL

Uses the presigned URL that LangSmith provides for each attachment. Simple and requires no extra API calls, but images must be under ~5 MB (Claude's limit for URL-referenced images).

In [4]:
def target_presigned_url(inputs: dict, attachments: dict) -> dict:
    """Approach 1: Use the presigned URL from LangSmith attachments.

    Pros: Simple, no extra API calls.
    Cons: Images must be under 5 MB (Claude's base64/URL limit).
    """
    client = anthropic.Anthropic()

    presigned_url = attachments["image"]["presigned_url"]

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "url",
                            "url": presigned_url,
                        },
                    },
                    {
                        "type": "text",
                        "text": inputs["input"],
                    },
                ],
            }
        ],
    )

    answer = next(
        (block.text for block in response.content if block.type == "text"), ""
    )
    return {"answer": answer}

### Approach 2: Anthropic Files API

Uploads the image to Anthropic's Files API first, then references it by `file_id`. Supports images up to 500 MB and avoids the base64 size overhead entirely. Requires the `files-api-2025-04-14` beta.

In [5]:
def target_files_api(inputs: dict, attachments: dict) -> dict:
    """Approach 2: Upload to Anthropic Files API, reference by file_id.

    Pros: Supports images up to 500 MB, avoids base64 overhead entirely.
    Cons: Extra upload step per image (could be cached/reused in production).
    """
    client = anthropic.Anthropic()

    image_reader = attachments["image"]["reader"]
    image_bytes = image_reader.read()
    mime_type = attachments["image"]["mime_type"]
    filename = inputs.get("image_file", "image.jpg")

    uploaded = client.beta.files.upload(
        file=(filename, image_bytes, mime_type),
    )

    try:
        response = client.beta.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1024,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {"type": "file", "file_id": uploaded.id},
                        },
                        {
                            "type": "text",
                            "text": inputs["input"],
                        },
                    ],
                }
            ],
            betas=["files-api-2025-04-14"],
        )
    finally:
        client.beta.files.delete(uploaded.id)

    answer = next(
        (block.text for block in response.content if block.type == "text"), ""
    )
    return {"answer": answer}

## LLM-as-Judge Evaluator

Uses Claude to score image descriptions on relevance, accuracy, detail, and coherence (0.0-1.0).

In [6]:
def llm_judge(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """LLM-as-judge evaluator using Claude to score image descriptions."""
    client = anthropic.Anthropic()

    prompt = f"""\
You are an expert evaluator assessing the quality of an image description.

The user asked: {inputs.get("input", "")}
The image file was: {inputs.get("image_file", "")}

Actual response:
{outputs.get("answer", "(no answer)")}

Expected response:
{reference_outputs.get("answer", "(no reference)")}

Evaluate the actual response on these criteria:
1. Relevance: Does the response actually answer the question asked?
2. Accuracy: Is the description consistent with what the expected output suggests?
3. Detail: Does the response provide a reasonable level of detail?
4. Coherence: Is the response well-structured and easy to understand?

First provide a brief assessment, then give a final score from 0.0 to 1.0.
You MUST end your response with exactly this format on the last line:
SCORE: <number between 0.0 and 1.0>"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )

    response_text = next(
        (block.text for block in response.content if block.type == "text"), ""
    )

    score = 0.5
    for line in reversed(response_text.strip().splitlines()):
        if line.strip().startswith("SCORE:"):
            try:
                score = float(line.strip().split("SCORE:")[1].strip())
                score = max(0.0, min(1.0, score))
            except ValueError:
                pass
            break

    return {
        "key": "image_description_quality",
        "score": score,
        "comment": response_text,
    }

## Helper: Print Results

In [7]:
def print_results(results):
    """Print experiment results in a readable format."""
    for result in results:
        example_input = result["example"].inputs
        run_output = result["run"].outputs or {}
        eval_results = result["evaluation_results"]["results"]
        print(f"\nImage: {example_input.get('image_file', 'N/A')}")
        print(f"Question: {example_input['input']}")
        answer = run_output.get("answer", "(error)")
        print(f"Answer: {answer[:150]}...")
        for er in eval_results:
            print(f"Score ({er.key}): {er.score}")
            if er.comment:
                print(f"Comment: {er.comment[:200]}")
        print("-" * 60)

## Experiment 1: Presigned URL Approach

In [8]:
results_presigned = ls_client.evaluate(
    target_presigned_url,
    data=DATASET_NAME,
    evaluators=[llm_judge],
    experiment_prefix="image-eval-presigned-url",
    description="Image descriptions via presigned URL (< 5 MB images)",
    max_concurrency=2,
)

print_results(results_presigned)

/Users/andrew/Desktop/customers/adobe/langsmith-large-image-evals/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'image-eval-presigned-url-50ea46f8' at:
https://smith.langchain.com/o/60995f7e-6a4c-4170-99c5-ad8ee020b23f/datasets/a2491b2a-d44d-4b87-bd61-8c9051d24383/compare?selectedSessions=1b1ac4fb-1d95-402e-8a94-91ed92d33ff8




5it [00:36,  7.31s/it]


Image: ocean_waves.jpg
Question: Describe the mood and atmosphere of this image.
Answer: # Mood and Atmosphere Analysis

This image evokes a **serene yet contemplative** mood through several key elements:

## Visual Qualities
- **Intimate ...
Score (image_description_quality): 0.92
Comment: ## Assessment

**Relevance (Excellent):**
The response directly addresses the user's question about mood and atmosphere. It provides a comprehensive analysis of the emotional and atmospheric qualities
------------------------------------------------------------

Image: desert_sunset.jpg
Question: Describe the lighting and time of day shown in this image.
Answer: # Lighting and Time of Day Analysis

This image appears to be taken during **late afternoon/golden hour**, likely approaching sunset. Here are the key...
Score (image_description_quality): 0.9
Comment: ## Assessment

**Relevance:** The response directly addresses the user's question about lighting and time of day. It provides comprehensive 

## Experiment 2: Files API Approach

In [ ]:
results_files_api = ls_client.evaluate(
    target_files_api,
    data=DATASET_NAME,
    evaluators=[llm_judge],
    experiment_prefix="image-eval-files-api",
    description="Image descriptions via Anthropic Files API (up to 500 MB images)",
    max_concurrency=2,
)

print_results(results_files_api)

View the evaluation results for experiment: 'image-eval-files-api-5ff09a9e' at:
https://smith.langchain.com/o/60995f7e-6a4c-4170-99c5-ad8ee020b23f/datasets/a2491b2a-d44d-4b87-bd61-8c9051d24383/compare?selectedSessions=97035924-f7a4-4a26-b884-fd26ab0ba4ce




0it [00:00, ?it/s]